# MLOps architecture on Databricks

This notebook is an overall architecture blueprint
This notebook is the overall architecture for the blueprint. I run the entire MLOps loop in miniature here, offline on synthetic churn data, so the whole shape sits in one place: raw data to features, training, the model registry, serving, and monitoring that feeds back into retraining. 

Every notebook that follows takes one stage of this loop and builds it for real on Databricks; the diagram below tags each stage with the notebook that does so. The exception is `06_agent_patterns`,
a separate track for retrieval-augmented agents that does not fit the churn loop.


**What it compares.**
1. The end-to-end reference architecture: raw data -> features -> train -> registry -> serve -> monitor.
2. Deploy-code vs deploy-model: what crosses the environment boundary.
3. The dev / staging / prod strategy that wraps the loop.

## The reference architecture

A single customer record moves through the loop: a raw row becomes features, trains a model, which
is registered to Unity Catalog, served to score new customers, and monitored for drift. When the
data shifts, monitoring triggers retraining and the loop repeats. Unity Catalog governs every stage
as the single boundary for data, features, models, and lineage.

```mermaid
graph LR
  RAW[Raw churn] --> FEAT[Features 03]
  FEAT --> TRAIN[Train + MLflow 03]
  TRAIN --> REG[UC registry 03]
  REG --> SERVE[Serve 03]
  SERVE --> MON[Monitoring 04]
  MON -->|drift triggers retrain| TRAIN
  UC[Unity Catalog governance] -.-> FEAT
  UC -.-> REG
  UC -.-> SERVE
```

The number on each stage is the notebook that builds it for real. Most of the loop, from features
through scoring, is one production pipeline in notebook `03`; monitoring is `04`. Config (`02`)
parameterizes that pipeline, and the deploy capstone (`05`) wraps it in dev/staging/prod targets and
CI/CD. Production serving endpoints are set up in `05`; here serving is a batch score.

## The loop in miniature

The cells below run every stage of the loop on synthetic data, offline. Each is intentionally
trivial; what matters is that one stage's output is the next stage's input. The order follows the
diagram: synthesize raw data, derive features, train a model, register it under a `@champion` alias,
score fresh customers, and check for drift.

In [ ]:
import numpy as np
import pandas as pd


def make_churn_data(n: int = 2000, seed: int = 42) -> pd.DataFrame:
    """Synthetic telco-churn frame, an offline stand-in for the IBM Telco dataset.

    Churn rises with month-to-month contracts, shorter tenure, and higher charges, so
    downstream models and drift metrics are non-trivial.
    """
    rng = np.random.default_rng(seed)
    contract = rng.choice(["month-to-month", "one-year", "two-year"], n, p=[0.55, 0.25, 0.20])
    tenure = rng.integers(1, 73, n)                          # account age in months
    monthly = rng.normal(70, 30, n).clip(18, 130).round(2)
    signal = (-0.3 + 1.3 * (contract == "month-to-month")
              - 0.04 * tenure + 0.015 * (monthly - 70))      # latent churn risk
    churn = (rng.random(n) < 1 / (1 + np.exp(-signal))).astype(int)
    return pd.DataFrame({
        "customer_id": [f"C{i:05d}" for i in range(n)],
        "contract": contract,
        "tenure": tenure,
        "monthly_charges": monthly,
        "churn": churn,
    })


raw = make_churn_data()
print(f"raw: {len(raw)} rows, churn rate {raw['churn'].mean():.1%}")
raw.head()

In [ ]:
FEATURES = ["is_month_to_month", "tenure", "monthly_charges", "charge_per_tenure"]


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Derive model-ready features from raw records. I/O-free and unit-testable, the shape the
    real PySpark transforms take in notebook 03."""
    out = df.copy()
    out["is_month_to_month"] = (out["contract"] == "month-to-month").astype(int)
    out["charge_per_tenure"] = (out["monthly_charges"] / out["tenure"]).round(3)
    return out


features = add_features(raw)
features[FEATURES + ["churn"]].head()

In [ ]:
def predict_proba(model: dict, X: np.ndarray) -> np.ndarray:
    """Score rows with a trained model dict: standardize, then apply the logistic."""
    Xs = (X - model["mean"]) / model["std"]
    return 1 / (1 + np.exp(-(Xs @ model["w"] + model["b"])))


def train_logreg(X: np.ndarray, y: np.ndarray, lr: float = 0.3, epochs: int = 500) -> dict:
    """Minimal logistic regression via gradient descent. Stands in for the MLflow-tracked
    training in notebook 03, which also logs params, metrics, and a model signature."""
    mean, std = X.mean(0), X.std(0)
    Xs = (X - mean) / std
    w, b = np.zeros(Xs.shape[1]), 0.0
    for _ in range(epochs):
        err = 1 / (1 + np.exp(-(Xs @ w + b))) - y
        w -= lr * (Xs.T @ err) / len(y)
        b -= lr * err.mean()
    return {"w": w, "b": b, "mean": mean, "std": std}


X = features[FEATURES].to_numpy(float)
y = features["churn"].to_numpy(float)
model = train_logreg(X, y)

accuracy = float(((predict_proba(model, X) > 0.5) == y).mean())   # a metric you would log to MLflow
print(f"trained logistic model, train accuracy {accuracy:.3f}")

In [ ]:
def uc_name(obj: str, *, catalog: str = "main", schema: str = "default") -> str:
    """Compose a Unity Catalog three-level name (catalog.schema.object). Defaults to main.default
    so examples run anywhere; per-environment catalogs just swap the first level."""
    return f"{catalog}.{schema}.{obj}"


# Notebook 03 registers to the real UC model registry. Here a dict shows the *shape*: a named,
# versioned model with a moveable alias (@champion points at the production version).
registry: dict = {}
aliases: dict = {}
model_name = uc_name("churn_model")
registry[model_name] = {1: model}
aliases[model_name] = {"champion": 1}
print(f"registered {model_name} v1; alias @champion -> v{aliases[model_name]['champion']}")

In [ ]:
# Serving: resolve the @champion version and score fresh customers. Notebook 05 deploys this
# behind a real Model Serving endpoint.
champion = registry[model_name][aliases[model_name]["champion"]]

new_customers = add_features(make_churn_data(n=300, seed=7))
new_customers["churn_risk"] = predict_proba(champion, new_customers[FEATURES].to_numpy(float)).round(3)
print(f"scored {len(new_customers)} customers; mean predicted risk {new_customers['churn_risk'].mean():.1%}")
new_customers[["customer_id", "tenure", "monthly_charges", "churn_risk"]].head()

In [ ]:
def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10) -> float:
    """Population Stability Index, a standard drift metric. >0.1 warrants a look, >0.2 is
    material drift. Notebook 04 computes this continuously with Lakehouse Monitoring."""
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected) + 1e-6
    a = np.histogram(actual, edges)[0] / len(actual) + 1e-6
    return float(np.sum((a - e) * np.log(a / e)))


drift = psi(features["monthly_charges"].to_numpy(), new_customers["monthly_charges"].to_numpy())
verdict = "stable" if drift < 0.1 else "investigate" if drift < 0.2 else "drift"
print(f"PSI(monthly_charges) = {drift:.4f} -> {verdict}")
# In production a 'drift' verdict trips the monitor->retrain edge, restarting the loop.

## Deploy-code vs deploy-model

The first architectural decision is what crosses the environment boundary from dev to prod.

- **Deploy-code (the blueprint default).** I promote the code. The training pipeline runs in each
  environment, so the model is retrained and registered in prod from prod data.
- **Deploy-model.** I promote the model artifact. Training runs once, usually in dev, and the
  resulting model is promoted through staging to prod.

```mermaid
graph TD
  CC[deploy-code: promote CODE] --> C1[dev: train]
  C1 --> C2[staging: train + test]
  C2 --> C3[prod: train + register]
  MM[deploy-model: promote ARTIFACT] --> M1[dev: train + register]
  M1 --> M2[staging: validate]
  M2 --> M3[prod: deploy artifact]
```

| Criterion | Favors deploy-code | Favors deploy-model |
|---|---|---|
| What is promoted / where training runs | Code; model trained in each env (incl. prod) | A model artifact; trained once, usually in dev |
| Retraining cadence | Frequent or automated retrains | Rare; expensive or manual training |
| Training-pipeline parity | Exercised in staging exactly as in prod | Training path not re-run downstream |
| Regulatory reproducibility | Pipeline reproducible from versioned code | Exact artifact pinned and promoted as-is |
| Training cost | Acceptable to retrain per environment | Very high; train once, reuse |
| Production data access | Prod data reachable for in-env training | Prod data restricted; train where data lives |

I default to deploy-code. It preserves train/serve parity and exercises the whole pipeline in
staging as prod runs it. I reach for deploy-model when training is too costly to repeat, or when
prod data cannot be reached from the training step.

## dev / staging / prod

The loop runs in three environments, separated by Unity Catalog catalog and by bundle target (the
deploy capstone, `05`). The code is identical across environments; only the catalog, and so the data
and models it sees, changes. Aliases are scoped to their catalog, so `@champion` in prod is
independent of `@champion` in dev.

| Environment | Catalog | Workspace | Data | Who promotes |
|---|---|---|---|---|
| dev | `dev` (or a sandbox) | shared dev | sampled / synthetic | any engineer |
| staging | `staging` | staging | prod-like, governed | CI on PR merge |
| prod | `prod` | prod (often isolated) | full production | gated CD / approver |

The runnable examples here use `main.default` so they work in any workspace with zero setup. In a
real deployment the catalog is swapped per environment via `uc_name(..., catalog=env)`, wired by the
bundle target.

## Further reading

[`docs/links.md`](docs/links.md) holds the full annotated list. For this notebook:

- [Big Book of MLOps](https://www.databricks.com/resources/ebook/the-big-book-of-mlops): the source of the deploy-code vs deploy-model framing.
- [MLOps on Databricks (recommended architecture)](https://docs.databricks.com/machine-learning/mlops/mlops-workflow.html): the canonical reference workflow.

The notebooks that follow build this loop for real: `02` config, `03` the training pipeline
(features, MLflow, registry, scoring), `04` monitoring and automation, and `05` deploy and promotion
with Asset Bundles.